# Hydromancer Jojo — MiniMax-H3 `Ref2VA` Side-by-Side Benchmark (Colab Pro L4)

Runs the **4 Reservoir handover shots (`j01`, `j02`, `j04`, `j05`)** locally on **Google Colab Pro (NVIDIA L4 — 24 GB VRAM)** using open-weight **MiniMax-H3 `Ref2VA`**, runs every automated QA check from `HANDOVER.md`, and renders a **labeled side-by-side comparison video** (**Replicate vs. Local Colab L4**) for Josh.

### What this notebook matches from `HANDOVER.md`
- **Exact 4-image reference order per shot:**
  1. `<Picture 1>`: glasses plate for that shot (`jojo-front-b.png` or `jojo-closeup.png`)
  2. `<Picture 2>`: `pose-gesture-right.png`
  3. `<Picture 3>`: `pose-present.png`
  4. `<Picture 4>`: `pose-count.png`
- **Exact reference audio (`<Audio 1>`):** `j01.wav` (4s), `j02.wav` (4s), `j04.wav` (4s), `j05.wav` (6s)
- **Resolution & FPS:** `1344×768` (16:9), `24 fps`, `17k+5` grid aligned (`107` frames for 4s, `158` frames for 6s)
- **Open-weight `Ref2VA` Prompt Formatting:** Formats Josh's exact `jNN.prompt.txt` into MiniMax-H3's official 6-section `Ref2VA` schema (`subject_definitions`, `summary`, `retention_analysis` with `<Audio 1>: fully_copy`, `detailed_description` with `<Picture 1>` & `<d>[English] ... </d>`, `overall_soundscape`, `non_diegetic_music`) to compensate for open weights not including hosted `H3-Context-IR`. (You can also toggle `raw_replicate_txt` to A/B test.)
- **Automated QA Checks:**
  - Scene-cut detection (`ffmpeg select='gt(scene,0.25)'`)
  - Audio cross-correlation sync offset (`measure_sync`)
  - Word-level lip-drift measurement + retiming via `faster-whisper` (Colab CUDA port of `tools/retime.py`)
  - Labeled side-by-side MP4 (`Hydromancer_H3_Replicate_vs_ColabL4.mp4`)

### Before Running
- Confirm **Runtime → Change runtime type → L4 GPU**.
- **License reminder for Josh (flagged, not decided):** Check both the MiniMax H3 Community License attribution/revenue clauses and the regional open-weight restriction (US/EU/UK/South Korea authorization form) before using open-weight outputs in production.

In [ ]:
#@title 1. Check L4 GPU, System RAM & Disk Space
!nvidia-smi
print("\n--- System RAM ---")
!free -h
print("\n--- Disk Space (/content) ---")
!df -h /content

In [ ]:
#@title 2. Download Hydromancer Handover Archive from Google Drive
import os

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1f4jIpG9BPBkv5k5oSnRwOrcAO5pfpZ3B?usp=sharing"  #@param {type:"string"}
HANDOVER_DIR = "/content/handover"

!pip install -q -U gdown

if not os.path.exists(os.path.join(HANDOVER_DIR, "examples/h3-reservoir/shots.json")):
    os.makedirs(HANDOVER_DIR, exist_ok=True)
    !gdown --folder "{DRIVE_FOLDER_URL}" -O "{HANDOVER_DIR}"
else:
    print("Handover folder already downloaded.")

assert os.path.exists(os.path.join(HANDOVER_DIR, "examples/h3-reservoir/shots.json")), "shots.json not found!"
print("Verified handover files:")
!ls -la /content/handover/examples/h3-reservoir/

In [ ]:
#@title 3. Install ComfyUI & QA Dependencies (faster-whisper, soundfile, opencv)
import os

%cd /content
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/Comfy-Org/ComfyUI.git
else:
    !git -C /content/ComfyUI pull

%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub[hf_transfer]" faster-whisper soundfile opencv-python-headless

# Optional cloudflared for Cell 9 interactive UI
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null
    !rm -f cloudflared-linux-amd64.deb

print("ComfyUI & QA tools installed.")

In [ ]:
#@title 4. Download L4-Optimized MiniMax-H3 Ref2VA Models (~37–42 GB)
import os
from huggingface_hub import hf_hub_download

#@markdown - `int8_convrot` (~21.0 GB): Official ComfyUI template default, best quality on L4.
#@markdown - `w6a8` (~16.0 GB): Smaller footprint if you hit VRAM pressure.
DIFFUSION_QUANT = "int8_convrot"  #@param ["int8_convrot", "w6a8"]
DOWNLOAD_TURBO_LORA = True  #@param {type:"boolean"}

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
REPO_ID = "Comfy-Org/MiniMax-H3"
MODELS_DIR = "/content/ComfyUI/models"

UNET_NAME = (
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
    if DIFFUSION_QUANT == "int8_convrot"
    else "minimax_h3_ref2va_pruned_w6a8.safetensors"
)
CLIP_NAME = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE_NAME = "minimax_h3_video_vae_int8_convrot.safetensors"
AUDIO_VAE_NAME = "minimax_h3_audio_vae_fp32.safetensors"
TURBO_LORA_NAME = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"

downloads = [
    (f"diffusion_models/{UNET_NAME}", f"{MODELS_DIR}/diffusion_models"),
    (f"text_encoders/{CLIP_NAME}", f"{MODELS_DIR}/text_encoders"),
    (f"vae/{VIDEO_VAE_NAME}", f"{MODELS_DIR}/vae"),
    (f"vae/{AUDIO_VAE_NAME}", f"{MODELS_DIR}/vae"),
]
if DOWNLOAD_TURBO_LORA:
    downloads.append((f"loras/{TURBO_LORA_NAME}", f"{MODELS_DIR}/loras"))

for repo_path, dest_dir in downloads:
    os.makedirs(dest_dir, exist_ok=True)
    fname = os.path.basename(repo_path)
    target = os.path.join(dest_dir, fname)
    if os.path.exists(target):
        print(f"[skip] {fname}")
        continue
    print(f"[downloading] {repo_path} ...")
    p = hf_hub_download(repo_id=REPO_ID, filename=repo_path, local_dir="/tmp/minimax_h3_cache")
    os.replace(p, target)
    print(f"[done] {target}")

print("\nAll MiniMax-H3 Ref2VA models ready.")

In [ ]:
#@title 5. Configure Benchmark Shots & Build Open-Weight Ref2VA Prompts
import json, os, shutil
from pathlib import Path

#@markdown ### Benchmark settings
#@markdown - `h3_ref_format`: Recommended for open weights (reproduces H3-Context-IR 6-section `<Picture 1>` / `<Audio 1>: fully_copy` / `<d>[English]...</d>` schema).
#@markdown - `raw_replicate_txt`: Verbatim `jNN.prompt.txt` with `<Picture 1>`..`<Picture 4>` & `<Audio 1>` tag mapping.
PROMPT_MODE = "h3_ref_format"  #@param ["h3_ref_format", "raw_replicate_txt"]
#@markdown - `USE_TURBO_4STEP = False` runs full 20 steps (closest quality comparison to Replicate).
#@markdown - `USE_TURBO_4STEP = True` runs 4 steps (~4-5x faster on L4, great for quick test).
USE_TURBO_4STEP = False  #@param {type:"boolean"}
SCHEDULER = "beta"  #@param ["beta", "simple", "normal"]
STEPS = 4 if USE_TURBO_4STEP else 20
#@markdown Run `j01` first for a fast 4s sample, or `j01,j02,j04,j05` for the full 4-shot benchmark.
SHOTS_TO_RUN = "j01"  #@param {type:"string"}
SEED = 4242  #@param {type:"integer"}

HANDOVER = Path("/content/handover")
H3_DIR = HANDOVER / "examples/h3-reservoir"
PLATES_DIR = HANDOVER / "refs/plates"
GESTURES_DIR = HANDOVER / "refs/gestures"
COMFY_INPUT = Path("/content/ComfyUI/input")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)

# Copy gesture refs (exact order from submit_h3.mjs)
GESTURE_REFS = ["pose-gesture-right.png", "pose-present.png", "pose-count.png"]
for g in GESTURE_REFS:
    shutil.copy2(GESTURES_DIR / g, COMFY_INPUT / g)

plan = json.loads((H3_DIR / "shots.json").read_text())
selected_ids = [x.strip() for x in SHOTS_TO_RUN.split(",") if x.strip()]
BENCH_SHOTS = [s for s in plan["shots"] if s["id"] in selected_ids]

def align_frame_count(seconds: int, fps: int = 24) -> int:
    n = max(5, round(seconds * fps))
    while n % 17 != 5:
        n += 1
    return n

def build_prompt(shot: dict, raw_prompt: str, mode: str) -> str:
    if mode == "raw_replicate_txt":
        txt = raw_prompt
        txt = txt.replace("reference image 1", "<Picture 1>")
        txt = txt.replace("Reference images 2-4", "<Picture 2>, <Picture 3>, and <Picture 4>")
        txt = txt.replace("Reference audio 1", "<Audio 1>")
        return txt

    # Official 6-section MiniMax-H3 Ref2VA format (VIDEO_PROMPT_WRITING_GUIDE_ref_en.md)
    framing = (
        "a centred front-on medium shot of <Subject 1> seated behind the wooden desk, with head, torso, and both paws visible"
        if shot["angle"] == "front-b"
        else "a centred front close-up shot of <Subject 1>, with head and shoulders filling the frame"
    )
    words = shot["words"]
    gesture = shot["gesture"]

    return (
        "subject_definitions:\n"
        "<Subject 1> is the stylized retro low-poly 3D Hydromancer cat mascot (Jojo) from <Picture 1>, "
        "with faceted angular geometry and painted textures (early-2000s game look), cream face, three grey forehead stripes, "
        "grey cheek stripes, cyan ear edges, pink inner ears, big glossy black eyes, navy hooded robe with bright cyan sleeve bands, "
        "blue water-droplet pendant with a small silver four-point star, and THICK BLACK-FRAMED GLASSES worn continuously on its face. "
        "<Picture 2>, <Picture 3>, and <Picture 4> show how <Subject 1>'s faceted low-poly paws look when gesturing (ignoring the absence of glasses in those gesture sheets).\n"
        "<Picture 1> is the exact composition, camera angle, framing, and real photographed studio anchor for [Shot 1], "
        "showing a warm glowing lamp on shelves, green plant, dark wall, framed black-and-white artwork, and wooden desk.\n"
        "<Audio 1> is the exact spoken narration audio track and timing for <Subject 1> (S1).\n\n"
        "summary:\n"
        "Task type: reference generation, audio reuse. "
        "The target video is a single uninterrupted static-camera studio presenter shot matching <Picture 1>, "
        "in which <Subject 1> (S1) speaks directly to camera with tight lip-sync to <Audio 1> and expressive low-poly paw gestures informed by <Picture 2>, <Picture 3>, and <Picture 4>.\n\n"
        "retention_analysis:\n"
        "<Subject 1> (appears in [Shot 1]): fully_preserved - exact retro low-poly faceted 3D cat identity, markings, navy hoodie robe, droplet pendant, and thick black-framed glasses stay locked on its face throughout; never smooth, realistic fur, whitish, extra ear tips, or hand-drawn.\n"
        "<Picture 1> ([Shot 1] composition and studio anchor): fully_preserved - exact camera angle, framing, proportions, and photorealistic real studio background remain completely unchanged with zero camera cuts, zooms, pans, or new props.\n"
        "<Audio 1>: fully_copy - <Audio 1> is reused 1:1 as the target video's complete final audio track and exact lip-sync timing.\n\n"
        "detailed_description:\n"
        "The target video blends a retro low-poly 3D character (<Subject 1>) seamlessly into a real photographed studio background from <Picture 1>.\n"
        f"[Shot 1] The camera holds a completely static, locked-off shot matching <Picture 1> verbatim: {framing}. "
        "The real photographed studio background (warm lamp, shelves, plant, dark wall, framed monochrome artwork, wooden desk) stays completely sharp, photorealistic, and unchanged—no camera movement, no zoom, no reframing, no cuts, no subtitles, no logos, and no text overlays. "
        "<Subject 1> (S1) remains seated behind the desk as a warm, charismatic presenter speaking directly to the viewer, with crisp low-poly facets and thick black-framed glasses staying fixed in place throughout. "
        f"Performance and gestures: {gesture} "
        "Between big gestures, <Subject 1> maintains subtle natural life—blinks, gentle breathing, tiny head tilts, and ear twitches. "
        "Gestures are clear and relaxed, never frantic; paws never cover the face or mouth; mouth moves strictly in sync with <Audio 1> and stays calmly closed during any silence without adding extra words. "
        f"<Subject 1> (S1) speaks with exact lip-sync to <Audio 1>: <d>[English] {words}</d>\n\n"
        "overall_soundscape:\n"
        "Clean studio voiceover copied 1:1 from <Audio 1>, with no added sound effects, no extra speech, and quiet studio silence when not speaking.\n\n"
        "non_diegetic_music:\n"
        "None."
    )

PREPARED_SHOTS = []
for s in BENCH_SHOTS:
    sid = s["id"]
    plate_name = os.path.basename(s["plate"])
    shutil.copy2(PLATES_DIR / plate_name, COMFY_INPUT / plate_name)
    wav_name = f"{sid}.wav"
    shutil.copy2(H3_DIR / wav_name, COMFY_INPUT / wav_name)
    raw_prompt = (H3_DIR / f"{sid}.prompt.txt").read_text().strip()
    final_prompt = build_prompt(s, raw_prompt, PROMPT_MODE)
    frames = align_frame_count(s["generationSeconds"])
    PREPARED_SHOTS.append({
        **s,
        "plate_name": plate_name,
        "wav_name": wav_name,
        "frames": frames,
        "prompt": final_prompt,
    })
    print(f"Prepared {sid}: plate={plate_name}, dur={s['generationSeconds']}s ({frames} frames @ 24fps), mode={PROMPT_MODE}, steps={STEPS}")

print("\nSample prompt (j01) preview:\n" + "-"*60)
print(PREPARED_SHOTS[0]["prompt"][:900] + "...")

In [ ]:
#@title 6. Start Headless ComfyUI & Generate All Selected Reservoir Shots
import json, os, shutil, subprocess, time, urllib.request
from pathlib import Path

BENCH_DIR = Path("/content/benchmark_out")
LOCAL_DIR = BENCH_DIR / "local"
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

COMFY_URL = "http://127.0.0.1:8188"

def comfy_is_up() -> bool:
    try:
        with urllib.request.urlopen(f"{COMFY_URL}/system_stats", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

# Start ComfyUI server in background if not already running
if not comfy_is_up():
    log_file = open("/content/comfyui_server.log", "w")
    comfy_proc = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    print("Starting ComfyUI server...")
    for _ in range(90):
        if comfy_is_up():
            break
        time.sleep(2)
    assert comfy_is_up(), "ComfyUI failed to start! Inspect /content/comfyui_server.log"
print("ComfyUI server is ready!")

def build_api_graph(shot: dict, seed: int) -> dict:
    model_ref = ["1", 0]
    graph = {
        "1": {
            "class_type": "UNETLoader",
            "inputs": {"unet_name": UNET_NAME, "weight_dtype": "default"},
        },
        "2": {
            "class_type": "CLIPLoader",
            "inputs": {"clip_name": CLIP_NAME, "type": "minimax", "device": "default"},
        },
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE_NAME}},
        "4": {"class_type": "VAELoader", "inputs": {"vae_name": AUDIO_VAE_NAME}},
        # 4 reference images in exact order (<Picture 1> .. <Picture 4>)
        "10": {"class_type": "LoadImage", "inputs": {"image": shot["plate_name"]}},
        "11": {"class_type": "LoadImage", "inputs": {"image": GESTURE_REFS[0]}},
        "12": {"class_type": "LoadImage", "inputs": {"image": GESTURE_REFS[1]}},
        "13": {"class_type": "LoadImage", "inputs": {"image": GESTURE_REFS[2]}},
        # Reference narration audio (<Audio 1>)
        "14": {"class_type": "LoadAudio", "inputs": {"audio": shot["wav_name"]}},
    }

    if USE_TURBO_4STEP:
        graph["5"] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {
                "model": ["1", 0],
                "lora_name": TURBO_LORA_NAME,
                "strength_model": 1.0,
            },
        }
        model_ref = ["5", 0]

    # IMPORTANT: ComfyUI HTTP API requires flat dotted keys indexed from 0
    # (ref_images.ref_image_0..3, ref_audios.ref_audio_0) so references are not silently dropped.
    graph.update({
        "20": {
            "class_type": "MiniMaxH3ReferenceToVideo",
            "inputs": {
                "clip": ["2", 0],
                "vae": ["3", 0],
                "audio_vae": ["4", 0],
                "prompt": shot["prompt"],
                "width": 1344,
                "height": 768,
                "length": shot["frames"],
                "ref_image_size": "match",
                "ref_images.ref_image_0": ["10", 0],
                "ref_images.ref_image_1": ["11", 0],
                "ref_images.ref_image_2": ["12", 0],
                "ref_images.ref_image_3": ["13", 0],
                "ref_audios.ref_audio_0": ["14", 0],
            },
        },
        "21": {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}},
        "22": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "res_multistep"}},
        "23": {
            "class_type": "BasicScheduler",
            "inputs": {"model": model_ref, "scheduler": SCHEDULER, "steps": STEPS, "denoise": 1.0},
        },
        "24": {
            "class_type": "BasicGuider",
            "inputs": {"model": model_ref, "conditioning": ["20", 0]},
        },
        "25": {
            "class_type": "SamplerCustomAdvanced",
            "inputs": {
                "noise": ["21", 0],
                "guider": ["24", 0],
                "sampler": ["22", 0],
                "sigmas": ["23", 0],
                "latent_image": ["20", 1],
            },
        },
        "26": {"class_type": "VAEDecode", "inputs": {"samples": ["25", 0], "vae": ["3", 0]}},
        "27": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["25", 0], "vae": ["4", 0]}},
        "28": {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "audio": ["27", 0], "fps": 24, "bit_depth": 8},
        },
        "29": {
            "class_type": "SaveVideo",
            "inputs": {
                "video": ["28", 0],
                "filename_prefix": f"hydromancer_bench/{shot['id']}",
                "format": "mp4",
                "codec": "h264",
            },
        },
    })
    return graph

def submit_and_wait(graph: dict) -> tuple[dict, float]:
    payload = json.dumps({"prompt": graph}).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFY_URL}/prompt", data=payload, headers={"Content-Type": "application/json"}
    )
    t0 = time.time()
    with urllib.request.urlopen(req) as r:
        res = json.loads(r.read().decode("utf-8"))
    pid = res["prompt_id"]
    while True:
        with urllib.request.urlopen(f"{COMFY_URL}/history/{pid}") as r:
            hist = json.loads(r.read().decode("utf-8"))
        if pid in hist:
            entry = hist[pid]
            status = entry.get("status", {})
            if status.get("status_str") == "error":
                raise RuntimeError(f"ComfyUI execution error: {json.dumps(status, indent=2)}")
            if entry.get("outputs") or status.get("completed"):
                return entry, time.time() - t0
        time.sleep(3)

GEN_TIMINGS = {}
for idx, shot in enumerate(PREPARED_SHOTS):
    sid = shot["id"]
    shot_seed = (SEED + idx) if SEED >= 0 else int(time.time() * 1000) % (2**31)
    print(f"\n=== Generating {sid} ({shot['generationSeconds']}s / {shot['frames']}f, steps={STEPS}, seed={shot_seed}) ===")
    graph = build_api_graph(shot, shot_seed)
    entry, elapsed = submit_and_wait(graph)

    # Locate saved MP4 from SaveVideo output
    out_29 = entry.get("outputs", {}).get("29", {})
    vids = out_29.get("images") or out_29.get("videos") or out_29.get("gifs") or []
    assert vids, f"No output video metadata found for {sid}: {out_29}"
    v = vids[0]
    src_mp4 = Path("/content/ComfyUI/output") / v.get("subfolder", "") / v["filename"]
    dst_mp4 = LOCAL_DIR / f"{sid}.mp4"
    shutil.copy2(src_mp4, dst_mp4)
    GEN_TIMINGS[sid] = round(elapsed, 1)
    print(f"[DONE] {sid} -> {dst_mp4} in {elapsed:.1f}s ({elapsed/shot['generationSeconds']:.1f}s per video sec)")

(BENCH_DIR / "timings.json").write_text(json.dumps({
    "gpu": "NVIDIA L4 (24GB)",
    "unet": UNET_NAME,
    "clip": CLIP_NAME,
    "prompt_mode": PROMPT_MODE,
    "turbo_4step": USE_TURBO_4STEP,
    "steps": STEPS,
    "scheduler": SCHEDULER,
    "timings_sec": GEN_TIMINGS,
}, indent=2))
print("\nAll shots generated! Timings:", GEN_TIMINGS)

In [ ]:
#@title 7. Run Automated QA Checks (Scene Cuts, Sync Offset, faster-whisper Retime)
import difflib, json, re, subprocess, tempfile
from pathlib import Path
import cv2
import numpy as np
import soundfile as sf
from faster_whisper import WhisperModel

BENCH_DIR = Path("/content/benchmark_out")
LOCAL_DIR = BENCH_DIR / "local"
RETIMED_DIR = BENCH_DIR / "local_retimed"
RETIMED_DIR.mkdir(parents=True, exist_ok=True)
H3_DIR = Path("/content/handover/examples/h3-reservoir")

def detect_scene_cuts(mp4_path: Path, threshold: float = 0.25) -> list[float]:
    cmd = [
        "ffmpeg", "-i", str(mp4_path),
        "-vf", f"select='gt(scene,{threshold})',showinfo",
        "-f", "null", "-",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    cuts = [float(m) for m in re.findall(r"pts_time:([0-9.]+)", proc.stderr)]
    return [round(t, 2) for t in cuts]

def measure_sync(mp4_path: Path, ref_wav_path: Path, fps: int = 24) -> dict:
    SR = 16000
    with tempfile.TemporaryDirectory() as tmp:
        c_wav = Path(tmp) / "clip.wav"
        r_wav = Path(tmp) / "ref.wav"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(mp4_path), "-vn", "-ac", "1", "-ar", str(SR), str(c_wav)], check=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(ref_wav_path), "-vn", "-ac", "1", "-ar", str(SR), str(r_wav)], check=True)
        clip = sf.read(c_wav)[0]
        ref = sf.read(r_wav)[0]
    n = min(len(clip), len(ref))
    a = clip[:n] - clip[:n].mean()
    b = ref[:n] - ref[:n].mean()
    c = np.correlate(a, b, "full")
    lag = (np.argmax(c) - (n - 1)) / SR
    sim = float(np.max(c) / np.sqrt((a ** 2).sum() * (b ** 2).sum() + 1e-9))
    return {
        "offset_ms": round(float(lag * 1000), 1),
        "shift_frames": int(-round(lag * fps)),
        "similarity": round(sim, 3),
    }

def retime_with_whisper(model: WhisperModel, clip_mp4: Path, ref_wav: Path, out_mp4: Path) -> dict:
    norm = lambda x: re.sub(r"[^a-z0-9]", "", x.lower())
    with tempfile.TemporaryDirectory() as tmp:
        c_wav = Path(tmp) / "clip.wav"
        r_wav = Path(tmp) / "ref.wav"
        v_only = Path(tmp) / "v.mp4"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(clip_mp4), "-vn", "-ac", "1", "-ar", "16000", str(c_wav)], check=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(ref_wav), "-vn", "-ac", "1", "-ar", "16000", str(r_wav)], check=True)

        def get_words(p):
            segs, _ = model.transcribe(str(p), language="en", word_timestamps=True, condition_on_previous_text=False)
            return [{"w": w.word, "s": float(w.start), "e": float(w.end)} for s in segs for w in (s.words or [])]

        narr, cw = get_words(r_wav), get_words(c_wav)
        a, b = [norm(w["w"]) for w in narr], [norm(w["w"]) for w in cw]
        anch = [(0.0, 0.0)]
        for m in difflib.SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks():
            for k in range(m.size):
                n_w, c_w = narr[m.a + k], cw[m.b + k]
                anch += [(n_w["s"], c_w["s"]), (n_w["e"], c_w["e"])]
        anch.sort()
        mono = [anch[0]]
        for p in anch[1:]:
            if p[0] > mono[-1][0] + 1e-3 and p[1] > mono[-1][1] + 1e-3:
                mono.append(p)
        nt, ct = np.array([p[0] for p in mono]), np.array([p[1] for p in mono])

        cap = cv2.VideoCapture(str(clip_mp4))
        fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
        W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frames = []
        while True:
            ok, fr = cap.read()
            if not ok:
                break
            frames.append(fr)
        cap.release()

        clip_dur = len(frames) / fps
        out_dur = float(nt[-1] + max(0.0, clip_dur - ct[-1]))
        ff = subprocess.Popen([
            "ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "bgr24",
            "-s", f"{W}x{H}", "-r", str(fps), "-i", "-",
            "-c:v", "libx264", "-crf", "14", "-pix_fmt", "yuv420p", str(v_only)
        ], stdin=subprocess.PIPE)
        for i in range(int(round(out_dur * fps))):
            t = i / fps
            c = float(np.interp(t, nt, ct)) if t <= nt[-1] else float(ct[-1] + (t - nt[-1]))
            ff.stdin.write(frames[min(len(frames) - 1, max(0, int(round(c * fps))))].tobytes())
        ff.stdin.close(); ff.wait()

        subprocess.run([
            "ffmpeg", "-v", "error", "-y", "-i", str(v_only), "-i", str(ref_wav),
            "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", str(out_mp4)
        ], check=True)
        drifts = [round(float(c - n), 3) for n, c in mono]
        return {
            "matched_anchors": len(mono),
            "max_drift_sec": max(drifts, key=abs) if drifts else 0.0,
            "ref_words": len(narr),
            "clip_words": len(cw),
        }

print("Loading faster-whisper (large-v3-turbo) on CPU/CUDA for lip-drift check...")
# Run Whisper on CPU (int8) so it doesn't collide with ComfyUI VRAM
whisper_model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8")

QA_REPORT = {}
for shot in PREPARED_SHOTS:
    sid = shot["id"]
    rep_mp4 = H3_DIR / f"{sid}.mp4"
    loc_mp4 = LOCAL_DIR / f"{sid}.mp4"
    ref_wav = H3_DIR / f"{sid}.wav"
    ret_mp4 = RETIMED_DIR / f"{sid}.mp4"

    rep_cuts = detect_scene_cuts(rep_mp4)
    loc_cuts = detect_scene_cuts(loc_mp4)
    rep_sync = measure_sync(rep_mp4, ref_wav)
    loc_sync = measure_sync(loc_mp4, ref_wav)
    retime_stats = retime_with_whisper(whisper_model, loc_mp4, ref_wav, ret_mp4)

    QA_REPORT[sid] = {
        "gen_time_sec": GEN_TIMINGS.get(sid),
        "replicate": {"cuts": rep_cuts, "sync": rep_sync},
        "local_l4": {"cuts": loc_cuts, "sync": loc_sync, "retime": retime_stats},
    }
    print(f"\n--- {sid} ({shot['generationSeconds']}s) ---")
    print(f"  Replicate : cuts={rep_cuts} | sync={rep_sync['offset_ms']:+.0f}ms (sim={rep_sync['similarity']})")
    print(f"  Local L4  : cuts={loc_cuts} | sync={loc_sync['offset_ms']:+.0f}ms (sim={loc_sync['similarity']}) | time={GEN_TIMINGS.get(sid)}s")
    print(f"  Retimer   : anchors={retime_stats['matched_anchors']} | max_drift={retime_stats['max_drift_sec']:+.3f}s")

(BENCH_DIR / "benchmark_report.json").write_text(json.dumps(QA_REPORT, indent=2))
print("\nSaved QA report -> /content/benchmark_out/benchmark_report.json")

In [ ]:
#@title 8. Render Side-by-Side Comparison Video for Josh & Download ZIP
import json, subprocess, zipfile
from pathlib import Path
from google.colab import files

#@markdown Choose which audio track plays in the side-by-side comparison video:
#@markdown - `local_raw`: Hear the exact raw audio generated by local MiniMax-H3 on Colab L4.
#@markdown - `ref_narration`: Hear Josh's clean ElevenLabs WAV (paired with the retimed local clip).
COMPARE_AUDIO = "local_raw"  #@param ["local_raw", "ref_narration"]

BENCH_DIR = Path("/content/benchmark_out")
H3_DIR = Path("/content/handover/examples/h3-reservoir")
SBS_DIR = BENCH_DIR / "side_by_side"
SBS_DIR.mkdir(parents=True, exist_ok=True)
FONT = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"

sbs_clips = []
for shot in PREPARED_SHOTS:
    sid = shot["id"]
    dur = shot["generationSeconds"]
    rep_mp4 = H3_DIR / f"{sid}.mp4"
    loc_mp4 = (BENCH_DIR / "local" / f"{sid}.mp4") if COMPARE_AUDIO == "local_raw" else (BENCH_DIR / "local_retimed" / f"{sid}.mp4")
    qa = QA_REPORT[sid]
    t_sec = qa["gen_time_sec"]
    l_cuts = len(qa["local_l4"]["cuts"])
    l_off = qa["local_l4"]["sync"]["offset_ms"]
    r_off = qa["replicate"]["sync"]["offset_ms"]

    left_label = f"{sid}  REPLICATE  (sync {r_off:+.0f}ms)"
    right_label = f"{sid}  LOCAL L4 {STEPS}st  ({t_sec}s | cuts {l_cuts} | sync {l_off:+.0f}ms)"
    esc = lambda s: s.replace(":", "\\:").replace("'", "\\'")

    out_sbs = SBS_DIR / f"{sid}_sbs.mp4"
    vf = (
        f"[0:v]scale=960:540:force_original_aspect_ratio=decrease,pad=960:590:0:50:#0F1A24,"
        f"drawtext=fontfile={FONT}:text='{esc(left_label)}':fontcolor=#F0F4F7:fontsize=22:x=20:y=14[left];"
        f"[1:v]scale=960:540:force_original_aspect_ratio=decrease,pad=960:590:0:50:#0F1A24,"
        f"drawtext=fontfile={FONT}:text='{esc(right_label)}':fontcolor=#97FCE4:fontsize=22:x=20:y=14[right];"
        f"[left][right]hstack=inputs=2[v]"
    )
    subprocess.run([
        "ffmpeg", "-v", "error", "-y",
        "-i", str(rep_mp4), "-i", str(loc_mp4),
        "-filter_complex", vf,
        "-map", "[v]", "-map", "1:a:0",
        "-t", str(dur), "-r", "24",
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "192k",
        str(out_sbs),
    ], check=True)
    sbs_clips.append(out_sbs)
    print("Built:", out_sbs)

concat_txt = SBS_DIR / "list.txt"
concat_txt.write_text("".join(f"file '{p}'\n" for p in sbs_clips))
combined_mp4 = BENCH_DIR / "Hydromancer_H3_Replicate_vs_ColabL4.mp4"
subprocess.run([
    "ffmpeg", "-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", str(concat_txt),
    "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k",
    str(combined_mp4),
], check=True)

bundle_zip = Path("/content/Hydromancer_H3_Benchmark_Results.zip")
with zipfile.ZipFile(bundle_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in BENCH_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(BENCH_DIR))

print(f"\nCombined side-by-side: {combined_mp4}")
print(f"Full benchmark bundle: {bundle_zip}")
files.download(str(combined_mp4))

In [ ]:
#@title 9. (Optional) Open ComfyUI Web UI via Cloudflare Tunnel to Inspect or Tweak Manually
import subprocess, re

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Open ComfyUI here:", m.group(0))
        break